In [1]:
import sys

In [2]:
from pyspark.sql import SparkSession


In [3]:
spark = (SparkSession
        .builder
        .appName("PythonMnMCount")
        .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/07 10:43:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Get the M&M data set file name

The dataset contains 100,000 lines, with each line being <state, color, count>. We want to compute and aggregate the counts for each color and state.

In [4]:
mnm_file = "data/mnm_dataset.csv"

In [5]:
mnm_df = (spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(mnm_file))

In [6]:
mnm_df.show(n=5, truncate=False)

+-----+-----+-----+
|State|Color|Count|
+-----+-----+-----+
|WY   |Brown|61   |
|ID   |Brown|34   |
|OR   |Green|47   |
|WY   |Red  |93   |
|ID   |Blue |94   |
+-----+-----+-----+
only showing top 5 rows


Aggregate count of all colors and groupBy state and color

orderBy descending order

In [7]:
count_mnm_df = (mnm_df.select("State", "Color", "Count")
                    .groupBy("State", "Color")
                    .sum("Count")
                    .orderBy("sum(Count)", ascending=False))

In [8]:
count_mnm_df.show(n=60, truncate=False)

+-----+------+----------+
|State|Color |sum(Count)|
+-----+------+----------+
|UT   |Blue  |97437     |
|WA   |Red   |95552     |
|UT   |Yellow|95333     |
|WA   |Brown |95258     |
|WY   |Blue  |95185     |
|OR   |Orange|94506     |
|ID   |Yellow|94441     |
|NV   |Orange|94338     |
|UT   |Brown |94242     |
|CO   |Red   |94177     |
|OR   |Blue  |94110     |
|WY   |Green |94105     |
|ID   |Green |94057     |
|AZ   |Orange|94039     |
|NM   |Brown |94015     |
|OR   |Red   |93896     |
|OR   |Yellow|93516     |
|WA   |Blue  |93253     |
|CA   |Brown |93183     |
|AZ   |Blue  |93062     |
|CA   |Red   |92788     |
|AZ   |Yellow|92787     |
|UT   |Red   |92727     |
|WA   |Orange|92722     |
|ID   |Brown |92579     |
|CA   |Blue  |92560     |
|CO   |Orange|92523     |
|NM   |Green |92478     |
|NM   |Red   |92366     |
|CA   |Orange|92312     |
|NM   |Yellow|92285     |
|WY   |Yellow|92116     |
|ID   |Blue  |91965     |
|OR   |Green |91775     |
|NV   |Yellow|91690     |
|WY   |Orang

In [9]:
print("Total Rows = %d" % (count_mnm_df.count()))

Total Rows = 60


The above code aggergated and counted for all the states. What if we just want to see the data for a single state. To do that: 
 1. Select from all rows in the DataFrame
 2. Filter only one state
 3. groupBy() State and Color as we did before
 4. Aggregate the counts for each color
 5. orderBy() in descending order
    

In [10]:
  ca_count_mnm_df = (mnm_df.select("*")
                       .where(mnm_df.State == 'CA')
                       .groupBy("State", "Color")
                       .sum("Count")
                       .orderBy("sum(Count)", ascending=False))

Show the resulting aggregation for California

In [11]:
ca_count_mnm_df.show(n=10, truncate=False)

+-----+------+----------+
|State|Color |sum(Count)|
+-----+------+----------+
|CA   |Brown |93183     |
|CA   |Red   |92788     |
|CA   |Blue  |92560     |
|CA   |Orange|92312     |
|CA   |Green |90476     |
|CA   |Yellow|89829     |
+-----+------+----------+

